In [1]:
from pathlib import Path
import subprocess, json, re, textwrap

repo = Path('/workspaces/rockstar-strudel')

# Run the full test suite
proc = subprocess.run(
    ['npm', 'test'],
    cwd=repo,
    capture_output=True,
    text=True,
)

output = (proc.stdout or '') + ('\n' + proc.stderr if proc.stderr else '')
print(output)

counts = {}
for key in ['tests', 'suites', 'pass', 'fail', 'cancelled', 'skipped', 'todo', 'duration_ms']:
    m = re.search(rf'#\s+{key}\s+(\d+)', output)
    if m:
        counts[key] = int(m.group(1))

failing_tests = []
for line in output.splitlines():
    stripped = line.strip()
    if stripped.startswith('not ok '):
        failing_tests.append(stripped[7:])

node_code = r'''
import * as mod from './src/index.js';
const result = {
  exports: Object.keys(mod).sort(),
  hasRockstar: typeof mod.rockstar === 'function',
  hasRockstarPro: typeof mod.rockstar_pro === 'function',
  parsedWordOutput: mod.parseOutputLine('hello world\n'),
  parsedNestedOutput: mod.parseOutputLine('["3", ["my dreams", "007"]]\n'),
  rockstarReturnsStateOutput: /return\s+state\.output\b/.test(String(mod.rockstar)),
  rockstarProExposesAliases:
    /output/.test(String(mod.rockstar_pro)) &&
    /mixed_output/.test(String(mod.rockstar_pro)) &&
    /text_output/.test(String(mod.rockstar_pro)) &&
    /raw_output/.test(String(mod.rockstar_pro)),
};
console.log(JSON.stringify(result));
'''
api_proc = subprocess.run(
    ['node', '--input-type=module', '-e', node_code],
    cwd=repo,
    capture_output=True,
    text=True,
)
print(api_proc.stdout)
if api_proc.stderr:
    print(api_proc.stderr)

api = json.loads(api_proc.stdout)

verification = {
    'pass_fail': 'PASS' if proc.returncode == 0 else 'FAIL',
    'summary_counts': counts,
    'failing_tests': failing_tests,
    'rockstar_numeric_first': bool(api['rockstarReturnsStateOutput']) and api['parsedWordOutput']['output'] == 55,
    'rockstar_pro_aliases': bool(api['rockstarProExposesAliases']),
    'api_exports': api['exports'],
}

print('\nFINAL_VERIFICATION_JSON=')
print(json.dumps(verification, indent=2))


> rockstar-strudel@1.0.4 test
> node --test test/*.test.js

▶ buildSource
  ✔ returns the raw template string when there are no interpolations (0.835688ms)
  ✔ splices a single string interpolation (0.159772ms)
  ✔ stringifies a numeric interpolation (0.114796ms)
  ✔ handles multiple interpolations in the correct order (0.129431ms)
  ✔ preserves leading and trailing whitespace / newlines in the template (0.217994ms)
  ✔ treats undefined interpolation values as empty strings (0.098519ms)
✔ buildSource (2.715025ms)
▶ coerce
  ✔ converts an integer string to a JS number (0.608473ms)
  ✔ converts a decimal string to a JS number (0.590633ms)
  ✔ converts a negative number string (2.175091ms)
  ✔ handles Windows-style CRLF line endings (0.363412ms)
  ✔ converts zero (0.158039ms)
  ✔ returns a string for non-numeric output (0.194609ms)
  ✔ returns a string for boolean-like output (0.094095ms)
  ✔ returns a string for null-like output (0.055956ms)
  ✔ returns undefined for an empty line (0.04